# Roboflow Ear Segmentation & Measurement

Uses Roboflow's general-segmentation-api-2 workflow to detect and measure ears from images.

In [ ]:
from inference_sdk import InferenceHTTPClient

# Initialize client and run workflow
client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key="R1zODk2Ja9JHBabfRASg"
)

path = "right.png"

result = client.run_workflow(
    workspace_name="fabiki4429-acoxs-com",
    workflow_id="general-segmentation-api-2",
    images={"image": path},
    parameters={"classes": "ear"},
    use_cache=True
)

## Extract Ear Measurements

In [ ]:
import numpy as np
import cv2

# Extract prediction data
prediction = result[0]['predictions']['predictions'][0]

# Get polygon points
points = np.array(
    [[p["x"], p["y"]] for p in prediction["points"]],
    dtype=np.int32
)

# Calculate bounding box dimensions
x_min, x_max = points[:, 0].min(), points[:, 0].max()
y_min, y_max = points[:, 1].min(), points[:, 1].max()

ear_width = x_max - x_min
ear_height = y_max - y_min

# Calculate area
area = cv2.contourArea(points)

print(f"Ear Width: {ear_width} px")
print(f"Ear Height: {ear_height} px")
print(f"Area: {area} px²")
print(f"Class: {prediction['class']}")
print(f"Confidence: {prediction['confidence']:.2f}")

## Generate Caliper Measurement Visualization

In [ ]:
import json
import math
from PIL import Image, ImageDraw
from IPython.display import display


def _extract_ear_points(api_result, class_name="ear"):
    """Extract polygon points from API result."""
    if isinstance(api_result, str):
        api_result = json.loads(api_result)
    
    if isinstance(api_result, list):
        api_result = api_result[0]
    
    predictions_block = api_result["predictions"]
    predictions = predictions_block["predictions"]
    
    match = next(
        (p for p in predictions if p["class"].strip() == class_name), None
    )
    if match is None:
        available = [p["class"] for p in predictions]
        raise ValueError(f"Class '{class_name}' not found. Available: {available}")
    
    points = [(p["x"], p["y"]) for p in match["points"]]
    model_w = predictions_block["image"]["width"]
    model_h = predictions_block["image"]["height"]
    
    return points, model_w, model_h


def _draw_capped_line(draw, p1, p2, color, width, cap_len):
    """Draw a line with perpendicular end-cap ticks (caliper style)."""
    draw.line([p1, p2], fill=color, width=width)
    
    dx, dy = p2[0] - p1[0], p2[1] - p1[1]
    length = math.hypot(dx, dy)
    if length == 0:
        return
    
    ux, uy = dx / length, dy / length
    px, py = -uy, ux
    half = cap_len / 2
    
    for cx, cy in (p1, p2):
        a = (cx - px * half, cy - py * half)
        b = (cx + px * half, cy + py * half)
        draw.line([a, b], fill=color, width=width)


def draw_ear_calipers(
    image_path,
    api_result,
    class_name="ear",
    pad_x_frac=0.55,
    pad_y_frac=0.20,
    width_line_height_frac=0.20,
    line_color=(255, 255, 255),
    outline_color=(230, 30, 30),
    line_width=2,
    cap_len=14,
):
    """Generate cropped caliper-style measurement visualization.
    
    Measurements include:
    - Vertical caliper: overall ear height
    - Horizontal caliper: width at upper region
    - Diagonal caliper: true top-to-bottom length
    """
    # Extract points and load image
    points, model_w, model_h = _extract_ear_points(api_result, class_name)
    img = Image.open(image_path).convert("RGB")
    W, H = img.size
    
    # Scale points to match image size
    scale_x, scale_y = W / model_w, H / model_h
    pts = [(x * scale_x, y * scale_y) for x, y in points]
    
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    ear_w_px, ear_h_px = max_x - min_x, max_y - min_y
    
    # Crop around ear with padding
    pad_x = ear_w_px * pad_x_frac
    pad_y = ear_h_px * pad_y_frac
    crop_box = (
        max(0, int(min_x - pad_x)),
        max(0, int(min_y - pad_y)),
        min(W, int(max_x + pad_x * 0.6)),
        min(H, int(max_y + pad_y)),
    )
    cropped = img.crop(crop_box)
    
    # Shift points to cropped coordinates
    ox, oy = crop_box[0], crop_box[1]
    pts_c = [(x - ox, y - oy) for x, y in pts]
    top, bottom = min_y - oy, max_y - oy
    left, right = min_x - ox, max_x - ox
    
    draw = ImageDraw.Draw(cropped)
    
    # Draw ear outline
    draw.line(pts_c + [pts_c[0]], fill=outline_color, width=2, joint="curve")
    
    # Vertical caliper (left of ear)
    vx = left - 18
    _draw_capped_line(draw, (vx, top), (vx, bottom), line_color, line_width, cap_len)
    
    # Horizontal caliper (upper region)
    hy = top + (bottom - top) * width_line_height_frac
    band = [p for p in pts_c if abs(p[1] - hy) < (bottom - top) * 0.05]
    if band:
        hx_left = min(p[0] for p in band)
        hx_right = max(p[0] for p in band)
    else:
        hx_left, hx_right = left, right
    _draw_capped_line(draw, (hx_left, hy), (hx_right, hy), line_color, line_width, cap_len)
    
    # Diagonal caliper (top to bottom)
    top_pt = min(pts_c, key=lambda p: p[1])
    bottom_pt = max(pts_c, key=lambda p: p[1])
    _draw_capped_line(draw, top_pt, bottom_pt, line_color, line_width, cap_len * 0.85)
    
    display(cropped)
    
    # Calculate diagonal length in original scale
    diag_len_px = math.hypot(
        (bottom_pt[0] - top_pt[0]) * (1 / scale_x if scale_x else 0),
        (bottom_pt[1] - top_pt[1]) * (1 / scale_y if scale_y else 0),
    ) * ((scale_x + scale_y) / 2)
    
    return {
        "ear_height_px": ear_h_px,
        "ear_width_top_px": hx_right - hx_left,
        "ear_diagonal_length_px": diag_len_px,
        "crop_box": crop_box,
    }

## Display Results

In [ ]:
measurements = draw_ear_calipers(path, result)

print("\n=== Measurements ===")
print(f"Height (px): {measurements['ear_height_px']:.1f}")
print(f"Width at top (px): {measurements['ear_width_top_px']:.1f}")
print(f"Diagonal length (px): {measurements['ear_diagonal_length_px']:.1f}")

## Other Visual Features of Your Ears

Subtle variations in ear structure, protrusions, and symmetry — drawn separately below:
- Gentle Ear Contour
- Backwards Ear Tilt
- Normal Ear Flare
- No Darwin's Tubercle

In [ ]:
from PIL import Image, ImageDraw
import math


def _load_scaled_points(image_path, api_result, class_name="ear"):
    """Loads image + rescales polygon points to the image's actual size."""
    points, model_w, model_h = _extract_ear_points(api_result, class_name)
    img = Image.open(image_path).convert("RGB")
    W, H = img.size
    scale_x, scale_y = W / model_w, H / model_h
    pts = [(x * scale_x, y * scale_y) for x, y in points]
    return img, pts


def _crop_around_ear(img, pts, pad_x_frac=0.55, pad_y_frac=0.20):
    """Crops the image tightly around the ear polygon and shifts points
    into the cropped image's coordinate space."""
    W, H = img.size
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    ear_w, ear_h = max_x - min_x, max_y - min_y

    pad_x, pad_y = ear_w * pad_x_frac, ear_h * pad_y_frac
    crop_box = (
        max(0, int(min_x - pad_x)),
        max(0, int(min_y - pad_y)),
        min(W, int(max_x + pad_x * 0.6)),
        min(H, int(max_y + pad_y)),
    )
    cropped = img.crop(crop_box)
    ox, oy = crop_box[0], crop_box[1]
    pts_c = [(x - ox, y - oy) for x, y in pts]
    return cropped, pts_c


def _dotted_circle(draw, center, radius, color, dot_count=28, dot_radius=1.6):
    """Draws a circle made of small dots (matches the reference marker style)."""
    cx, cy = center
    for i in range(dot_count):
        theta = 2 * math.pi * i / dot_count
        dx, dy = cx + radius * math.cos(theta), cy + radius * math.sin(theta)
        draw.ellipse(
            [dx - dot_radius, dy - dot_radius, dx + dot_radius, dy + dot_radius],
            fill=color,
        )


# ---------------------------------------------------------------------------
# 1. Gentle Ear Contour — dotted circle marking the lobe/lower contour area
# ---------------------------------------------------------------------------
def draw_gentle_ear_contour(image_path, api_result, class_name="ear",
                            color=(255, 255, 255)):
    img, pts = _load_scaled_points(image_path, api_result, class_name)
    cropped, pts_c = _crop_around_ear(img, pts)

    xs, ys = [p[0] for p in pts_c], [p[1] for p in pts_c]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    ear_w, ear_h = max_x - min_x, max_y - min_y

    # Lower lobe/contour region, near the front-lower edge of the ear
    center = (min_x + ear_w * 0.30, min_y + ear_h * 0.78)
    radius = ear_w * 0.16

    draw = ImageDraw.Draw(cropped)
    _dotted_circle(draw, center, radius, color)
    return cropped


# ---------------------------------------------------------------------------
# 2. Backwards Ear Tilt — ear's central axis vs. a true-vertical reference
# ---------------------------------------------------------------------------
def draw_backwards_ear_tilt(image_path, api_result, class_name="ear",
                             color=(255, 255, 255), width=2):
    img, pts = _load_scaled_points(image_path, api_result, class_name)
    cropped, pts_c = _crop_around_ear(img, pts)

    top_pt = min(pts_c, key=lambda p: p[1])
    bottom_pt = max(pts_c, key=lambda p: p[1])

    draw = ImageDraw.Draw(cropped)
    # Both lines share the same start point: the bottom tip/edge of the ear
    # Ear's actual axis (bottom tip -> top of ear)
    draw.line([bottom_pt, top_pt], fill=color, width=width)
    # True-vertical reference line rising straight up from the bottom tip
    vertical_top = (bottom_pt[0], top_pt[1])
    draw.line([bottom_pt, vertical_top], fill=color, width=width)

    dx, dy = bottom_pt[0] - top_pt[0], bottom_pt[1] - top_pt[1]
    tilt_deg = math.degrees(math.atan2(dx, dy))
    return cropped, tilt_deg


# ---------------------------------------------------------------------------
# 3. Normal Ear Flare — traces the curved top edge/rim of the ear, showing
#    how far it flares outward from the head
# ---------------------------------------------------------------------------
def draw_normal_ear_flare(image_path, api_result, class_name="ear",
                           color=(255, 255, 255), width=3):
    img, pts = _load_scaled_points(image_path, api_result, class_name)
    cropped, pts_c = _crop_around_ear(img, pts)

    xs, ys = [p[0] for p in pts_c], [p[1] for p in pts_c]
    min_y, max_y = min(ys), max(ys)
    ear_h = max_y - min_y

    # Top edge = points in the upper band of the ear, sorted left -> right
    top_band = sorted(
        [p for p in pts_c if p[1] <= min_y + ear_h * 0.20],
        key=lambda p: p[0]
    )

    draw = ImageDraw.Draw(cropped)
    draw.line(top_band, fill=color, width=width, joint="curve")
    return cropped


# ---------------------------------------------------------------------------
# 4. No Darwin's Tubercle — marks the upper helix rim where a tubercle
#    (a small point/bump) would appear if present
# ---------------------------------------------------------------------------
def draw_darwins_tubercle_check(image_path, api_result, class_name="ear",
                                 color=(255, 255, 255)):
    img, pts = _load_scaled_points(image_path, api_result, class_name)
    cropped, pts_c = _crop_around_ear(img, pts)

    xs, ys = [p[0] for p in pts_c], [p[1] for p in pts_c]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    ear_w, ear_h = max_x - min_x, max_y - min_y

    # Upper-front rim, where the helix attaches near the top of the ear
    center = (min_x + ear_w * 0.12, min_y + ear_h * 0.10)
    radius = ear_w * 0.14

    draw = ImageDraw.Draw(cropped)
    _dotted_circle(draw, center, radius, color)
    return cropped


# --- Run each one separately ---
print("1. Gentle Ear Contour")
display(draw_gentle_ear_contour(path, result))

print("2. Backwards Ear Tilt")
tilt_img, tilt_angle = draw_backwards_ear_tilt(path, result)
display(tilt_img)
print(f"Tilt angle from vertical: {tilt_angle:.1f}°")

print("3. Normal Ear Flare")
display(draw_normal_ear_flare(path, result))

print("4. No Darwin's Tubercle")
display(draw_darwins_tubercle_check(path, result))

## Full Ear Measurement Diagram

A single composite figure combining overall height, overall width, upper-ear
reference line, concha width, diagonal axis, and lobe width — clinical
caliper style.

In [ ]:
def _bracket(draw, p1, p2, tick_vec, color, width, tick_len):
    """Draws a line from p1 to p2 with a short perpendicular tick at each
    end, pointing in the direction of tick_vec (a unit vector)."""
    draw.line([p1, p2], fill=color, width=width)
    tx, ty = tick_vec
    for p in (p1, p2):
        end = (p[0] + tx * tick_len, p[1] + ty * tick_len)
        draw.line([p, end], fill=color, width=width)


def _dashed_line(draw, p1, p2, color, width=2, dash_len=6, gap_len=5):
    """Draws a horizontal/vertical/diagonal dashed line from p1 to p2."""
    x1, y1 = p1
    x2, y2 = p2
    length = math.hypot(x2 - x1, y2 - y1)
    if length == 0:
        return
    ux, uy = (x2 - x1) / length, (y2 - y1) / length
    dist = 0
    while dist < length:
        seg_start = (x1 + ux * dist, y1 + uy * dist)
        seg_end_dist = min(dist + dash_len, length)
        seg_end = (x1 + ux * seg_end_dist, y1 + uy * seg_end_dist)
        draw.line([seg_start, seg_end], fill=color, width=width)
        dist += dash_len + gap_len


def draw_full_ear_measurements(image_path, api_result, class_name="ear",
                                color=(255, 255, 255), width=2, tick_len=12):
    """Composite measurement figure: overall height, overall width, an upper
    dotted reference line, concha width, a diagonal tilt-axis line, and a
    small lobe-width bracket — all in one image."""
    img, pts = _load_scaled_points(image_path, api_result, class_name)
    cropped, pts_c = _crop_around_ear(img, pts, pad_x_frac=0.75, pad_y_frac=0.45)

    xs, ys = [p[0] for p in pts_c], [p[1] for p in pts_c]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    ear_w, ear_h = max_x - min_x, max_y - min_y

    draw = ImageDraw.Draw(cropped)

    # 1. Overall height — vertical bracket to the left of the ear
    vx = min_x - 30
    _bracket(draw, (vx, min_y), (vx, max_y), tick_vec=(1, 0),
             color=color, width=width, tick_len=tick_len)

    # 2. Overall width — horizontal bracket above the ear, legs dropping
    #    down to the ear's left/right extremes
    hy = min_y - 25
    _bracket(draw, (min_x, hy), (max_x, hy), tick_vec=(0, 1),
             color=color, width=width, tick_len=min_y - hy)

    # 3. Upper-ear dotted reference line, just below the width bracket
    dot_y = min_y + ear_h * 0.06
    _dashed_line(draw, (min_x, dot_y), (max_x, dot_y), color=color, width=width)

    # 4. Concha (middle) width — horizontal bracket through the ear's center
    mid_y = min_y + ear_h * 0.55
    mx1, mx2 = min_x + ear_w * 0.25, max_x - ear_w * 0.08
    _bracket(draw, (mx1, mid_y), (mx2, mid_y), tick_vec=(0, -1),
             color=color, width=width, tick_len=tick_len * 0.7)

    # 5. Diagonal tilt axis — top of ear to lower-outer edge, with a small
    #    bracket at the bottom end
    top_pt = min(pts_c, key=lambda p: p[1])
    diag_end = (max_x - ear_w * 0.05, max_y)
    draw.line([top_pt, diag_end], fill=color, width=width)
    dx, dy = diag_end[0] - top_pt[0], diag_end[1] - top_pt[1]
    seg_len = math.hypot(dx, dy)
    perp = (-dy / seg_len, dx / seg_len) if seg_len else (0, 0)
    half = tick_len * 0.6
    draw.line([
        (diag_end[0] - perp[0] * half, diag_end[1] - perp[1] * half),
        (diag_end[0] + perp[0] * half, diag_end[1] + perp[1] * half),
    ], fill=color, width=width)

    # 6. Lobe width — small bracket near the bottom of the ear
    lobe_y = max_y - ear_h * 0.12
    lx1, lx2 = min_x + ear_w * 0.35, max_x - ear_w * 0.30
    _bracket(draw, (lx1, lobe_y), (lx2, lobe_y), tick_vec=(0, -1),
             color=color, width=width, tick_len=tick_len * 0.6)

    return cropped


display(draw_full_ear_measurements(path, result))